In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║   SKIN CANCER 9-CLASS ISIC — Q1 JOURNAL MODEL                              ║
# ║   Dataset : kaggle/nodoubttome/skin-cancer9-classesisic                     ║
# ║   Phase 1 : Single Train  → All Journal Figures                            ║
# ║   Phase 2 : 5-Fold CV     → All Journal Figures                            ║
# ║   Phase 3 : Explainable AI (Grad-CAM, LIME, SHAP)                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ========================== INSTALL ==========================
# !pip install -q albumentations==1.4.16 grad-cam lime shap

# ========================== IMPORTS ==========================
import os, warnings, copy, json, random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.patches import Rectangle

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, roc_curve, auc, cohen_kappa_score
)

# ========================== SEED ==========================
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ========================== DEVICE ==========================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

# ========================== PATHS ==========================
BASE_DIR   = "/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration"
TRAIN_DIR  = os.path.join(BASE_DIR, "Train")
TEST_DIR   = os.path.join(BASE_DIR, "Test")

# Output dirs
P1_DIR  = "./output/phase1"
P2_DIR  = "./output/phase2"
XAI_DIR = "./output/xai"
for d in [P1_DIR, P2_DIR, XAI_DIR]:
    os.makedirs(d, exist_ok=True)

# ========================== 9 CLASSES ==========================
# Folder names discovered from ImageFolder structure
CLASS_NAMES = sorted([
    d for d in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, d))
])
NUM_CLASSES  = len(CLASS_NAMES)
CLASS_ABBR   = [c[:6].upper() for c in CLASS_NAMES]

# Readable full names (map by common ISIC 9-class labels)
FULL_NAME_MAP = {
    'melanoma':            'Melanoma',
    'melanocytic_nevi':    'Melanocytic Nevi',
    'basal_cell_carcinoma':'Basal Cell Carcinoma',
    'actinic_keratosis':   'Actinic Keratosis',
    'benign_keratosis':    'Benign Keratosis',
    'dermatofibroma':      'Dermatofibroma',
    'vascular_lesion':     'Vascular Lesion',
    'squamous_cell_carcinoma': 'Squamous Cell Carcinoma',
    'seborrheic_keratosis':'Seborrheic Keratosis',
}
CLASS_FULL = [FULL_NAME_MAP.get(c, c.replace('_',' ').title()) for c in CLASS_NAMES]
CLASS_COLORS = [
    '#E63946','#457B9D','#2A9D8F','#E9C46A','#F4A261',
    '#264653','#A8DADC','#6A4C93','#F77F00'
]

print(f"\n9 Classes detected: {CLASS_NAMES}")

# ========================== JOURNAL STYLE ==========================
plt.rcParams.update({
    'font.family':        'serif',
    'font.serif':         ['Times New Roman','DejaVu Serif'],
    'font.size':          11,
    'axes.titlesize':     13,
    'axes.labelsize':     12,
    'xtick.labelsize':    10,
    'ytick.labelsize':    10,
    'legend.fontsize':    10,
    'figure.dpi':         300,
    'savefig.dpi':        300,
    'savefig.bbox':       'tight',
    'savefig.pad_inches': 0.1,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          True,
    'grid.alpha':         0.3,
    'grid.linestyle':     '--',
})

# ============================================================
# BUILD META DATAFRAMES from folder structure
# ============================================================
def build_meta(root_dir, split_name):
    records = []
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        cls_dir = os.path.join(root_dir, cls_name)
        if not os.path.isdir(cls_dir):
            continue
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith(('.jpg','.jpeg','.png')):
                records.append({
                    'path':     os.path.join(cls_dir, fname),
                    'label':    cls_idx,
                    'class':    cls_name,
                    'split':    split_name,
                })
    return pd.DataFrame(records)

train_meta_full = build_meta(TRAIN_DIR, 'Train')
test_meta       = build_meta(TEST_DIR,  'Test')

print(f"\nTotal Train: {len(train_meta_full)} | Test: {len(test_meta)}")
print("\nTrain class distribution:")
print(train_meta_full['class'].value_counts())

# ========================== TRANSFORMS ==========================
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=25, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30,
                          val_shift_limit=15, p=0.4),
    A.GaussianBlur(blur_limit=(3,5), p=0.2),
    A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(8, 16), hole_width_range=(8, 16), p=0.3),
    A.CLAHE(clip_limit=(1.0, 4.0), p=0.3),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])

# ========================== DATASET ==========================
class SkinDataset(Dataset):
    def __init__(self, meta_df, transform=None):
        self.meta      = meta_df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.meta)

    def __getitem__(self, idx):
        row   = self.meta.iloc[idx]
        image = np.array(Image.open(row['path']).convert('RGB'))
        if self.transform:
            image = self.transform(image=image)['image']
        return image, torch.tensor(int(row['label']), dtype=torch.long)

# ========================== WEIGHTED SAMPLER ==========================
def get_sampler(meta_df):
    counts   = np.bincount(meta_df['label'].values, minlength=NUM_CLASSES)
    weights  = 1.0 / np.maximum(counts, 1)
    s_weights= weights[meta_df['label'].values]
    return WeightedRandomSampler(
        torch.FloatTensor(s_weights), len(s_weights), replacement=True
    )

def get_loss_weights(meta_df):
    counts = np.bincount(meta_df['label'].values, minlength=NUM_CLASSES).astype(float)
    w = counts.sum() / (NUM_CLASSES * np.maximum(counts, 1))
    return torch.tensor(np.clip(w, 0.3, 8.0), dtype=torch.float32).to(device)

# ========================== MODEL ==========================
class SkinCancerModel(nn.Module):
    """
    EfficientNet-B2 backbone (stronger than B0 for 9-class task).
    Dropout regularization + BatchNorm fusion head.
    """
    def __init__(self, num_classes=9, dropout=0.4):
        super().__init__()
        self.backbone = models.efficientnet_b2(pretrained=True)
        in_feat = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()

        self.head = nn.Sequential(
            nn.BatchNorm1d(in_feat),
            nn.Dropout(dropout),
            nn.Linear(in_feat, 512),
            nn.SiLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout / 2),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.head(self.backbone(x))

    def get_features(self, x):
        return self.backbone(x)

# ========================== TRAIN / EVAL ==========================
def train_epoch(model, loader, criterion, optimizer, scheduler_warmup=None):
    model.train()
    total_loss = correct = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler_warmup: scheduler_warmup.step()
        total_loss += loss.item()
        correct    += (out.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate_model(model, loader):
    model.eval()
    preds, labels, probs = [], [], []
    with torch.no_grad():
        for imgs, lbl in loader:
            imgs = imgs.to(device)
            out  = model(imgs)
            prob = torch.softmax(out, 1)
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(lbl.numpy())
            probs.extend(prob.cpu().numpy())
    return np.array(labels), np.array(preds), np.array(probs)

# ============================================================
# SHARED FIGURE FUNCTIONS (used in both Phase 1 & 2)
# ============================================================

def save_fig1_distribution(meta_df, out_dir, prefix="fig1"):
    counts = meta_df['class'].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

    bars = axes[0].bar(CLASS_ABBR, counts.values,
                       color=CLASS_COLORS, edgecolor='white', linewidth=0.8, alpha=0.9)
    for bar, val in zip(bars, counts.values):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+8,
                     f'{val:,}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    axes[0].set_xlabel("Lesion Class", fontweight='bold')
    axes[0].set_ylabel("Sample Count",  fontweight='bold')
    axes[0].set_title("(a) Sample Count per Class", fontweight='bold')
    axes[0].set_facecolor('#F8F9FA')
    axes[0].tick_params(axis='x', rotation=30)

    wedges, texts, autotexts = axes[1].pie(
        counts.values, labels=CLASS_ABBR, colors=CLASS_COLORS,
        autopct='%1.1f%%', startangle=140,
        wedgeprops={'edgecolor':'white','linewidth':1.2}
    )
    for at in autotexts: at.set_fontsize(8)
    axes[1].set_title("(b) Class Proportion", fontweight='bold')

    fig.suptitle("Figure 1: Class Distribution — 9-Class ISIC Skin Cancer Dataset",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_class_distribution.png"), dpi=300)
    plt.close()
    print(f"  ✔ {prefix} class distribution saved")


def save_fig2_augmentation(out_dir, prefix="fig2"):
    # pick one sample per class
    sample_paths = []
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(TRAIN_DIR, cls)
        files   = [f for f in os.listdir(cls_dir)
                   if f.lower().endswith(('.jpg','.jpeg','.png'))]
        if files:
            sample_paths.append((cls, os.path.join(cls_dir, files[0])))

    aug_ops = [
        ("Original",            A.Compose([A.Resize(112,112)])),
        ("Horiz. Flip",         A.Compose([A.Resize(112,112), A.HorizontalFlip(p=1)])),
        ("Vert. Flip",          A.Compose([A.Resize(112,112), A.VerticalFlip(p=1)])),
        ("Rotate 90°",          A.Compose([A.Resize(112,112), A.RandomRotate90(p=1)])),
        ("Shift/Scale/Rot",     A.Compose([A.Resize(112,112), A.ShiftScaleRotate(
                                    shift_limit=0.15, scale_limit=0.15,
                                    rotate_limit=30, p=1)])),
        ("Bright/Contrast",     A.Compose([A.Resize(112,112),
                                    A.RandomBrightnessContrast(0.4, 0.4, p=1)])),
        ("Hue/Saturation",      A.Compose([A.Resize(112,112),
                                    A.HueSaturationValue(20,40,20,p=1)])),
        ("Coarse Dropout",      A.Compose([A.Resize(112,112),
                                    A.CoarseDropout(num_holes_range=(1,10), hole_height_range=(8,18), hole_width_range=(8,18), p=1)])),
        ("CLAHE",               A.Compose([A.Resize(112,112), A.CLAHE(clip_limit=(2.0,8.0), p=1)])),
    ]

    nrows = min(len(sample_paths), 4)  # show first 4 classes
    ncols = len(aug_ops)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*1.8, nrows*2.0))

    for r, (cls_name, path) in enumerate(sample_paths[:nrows]):
        img = np.array(Image.open(path).convert("RGB"))
        for c, (op_name, tfm) in enumerate(aug_ops):
            aug = tfm(image=img)['image']
            axes[r][c].imshow(aug)
            axes[r][c].axis('off')
            if r == 0:
                axes[r][c].set_title(op_name, fontsize=7.5, fontweight='bold', pad=3)
            if c == 0:
                axes[r][c].set_ylabel(cls_name.replace('_','\n'), fontsize=7,
                                      rotation=0, labelpad=55, va='center')

    fig.suptitle("Figure 2: Augmentation Pipeline — 4 Sample Classes × 9 Techniques",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_augmentation_grid.png"), dpi=300)
    plt.close()
    print(f"  ✔ {prefix} augmentation grid saved")


def save_fig_training_curves(history, out_dir, prefix="fig3"):
    ep = range(1, len(history['train_loss'])+1)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].plot(ep, history['train_loss'], color='#E63946', lw=2, label='Train')
    axes[0].plot(ep, history['val_loss'],   color='#457B9D', lw=2, ls='--', label='Val')
    axes[0].set_xlabel("Epoch", fontweight='bold')
    axes[0].set_ylabel("Loss",  fontweight='bold')
    axes[0].set_title("(a) Loss Curves", fontweight='bold')
    axes[0].legend(); axes[0].set_facecolor('#F8F9FA')

    axes[1].plot(ep, history['train_acc'], color='#E63946', lw=2, label='Train Acc')
    axes[1].plot(ep, history['val_acc'],   color='#2A9D8F', lw=2, ls='--', label='Val Acc')
    best_ep = int(np.argmax(history['val_acc'])) + 1
    axes[1].axvline(best_ep, color='gray', ls=':', lw=1.5, label=f'Best Ep={best_ep}')
    axes[1].set_xlabel("Epoch", fontweight='bold')
    axes[1].set_ylabel("Accuracy", fontweight='bold')
    axes[1].set_title("(b) Accuracy Curves", fontweight='bold')
    axes[1].legend(); axes[1].set_facecolor('#F8F9FA')

    val_arr = np.array(history['val_acc'])
    axes[2].plot(ep, val_arr, color='#264653', lw=2)
    axes[2].fill_between(ep, val_arr, alpha=0.15, color='#264653')
    best_acc = val_arr.max()
    axes[2].axhline(best_acc, color='#E63946', ls='--', lw=1.5,
                    label=f'Best = {best_acc:.4f}')
    axes[2].set_xlabel("Epoch", fontweight='bold')
    axes[2].set_ylabel("Val Accuracy", fontweight='bold')
    axes[2].set_title("(c) Validation Progression", fontweight='bold')
    axes[2].legend(); axes[2].set_facecolor('#F8F9FA')

    title_label = "Phase 1 — " if "p1" in prefix else "Phase 2 (Mean ± SD) — "
    fig.suptitle(f"Figure 3: {title_label}Training & Validation Curves",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_training_curves.png"), dpi=300)
    plt.close()
    print(f"  ✔ {prefix} training curves saved")


def save_fig_confusion_matrix(true, pred, out_dir, prefix="fig4"):
    cm      = confusion_matrix(true, pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
    for ax, data, fmt, title, cmap in zip(
        axes, [cm, cm_norm], ['d', '.2f'],
        ['(a) Raw Counts', '(b) Normalized (Recall per Class)'],
        ['Blues','RdYlGn']
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap=cmap,
                    xticklabels=CLASS_ABBR, yticklabels=CLASS_ABBR,
                    linewidths=0.4, linecolor='white', ax=ax,
                    cbar_kws={'shrink':0.75}, annot_kws={'fontsize':8})
        ax.set_xlabel("Predicted", fontweight='bold', labelpad=8)
        ax.set_ylabel("True",      fontweight='bold', labelpad=8)
        ax.set_title(title, fontweight='bold', pad=8)
        ax.tick_params(axis='x', rotation=45, labelsize=8)
        ax.tick_params(axis='y', rotation=0,  labelsize=8)

    tag = "Phase 1" if "p1" in prefix else "Phase 2 (OOF)"
    fig.suptitle(f"Figure 4: Confusion Matrix — {tag} Test Set",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_confusion_matrix.png"), dpi=300)
    plt.close()
    print(f"  ✔ {prefix} confusion matrix saved")


def save_fig_roc_curves(true, probs, out_dir, prefix="fig5"):
    y_bin = label_binarize(true, classes=list(range(NUM_CLASSES)))
    ncols = 5; nrows = 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 7))
    axes = axes.flatten()
    auc_vals = {}

    for i, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_bin[:,i], probs[:,i])
        roc_auc = auc(fpr, tpr)
        auc_vals[cls] = roc_auc
        axes[i].plot(fpr, tpr, color=CLASS_COLORS[i], lw=2.5,
                     label=f'AUC={roc_auc:.3f}')
        axes[i].plot([0,1],[0,1],'k--', lw=1, alpha=0.4)
        axes[i].fill_between(fpr, tpr, alpha=0.08, color=CLASS_COLORS[i])
        axes[i].set_xlabel("FPR", fontsize=8)
        axes[i].set_ylabel("TPR", fontsize=8)
        axes[i].set_title(CLASS_ABBR[i], fontweight='bold', fontsize=9)
        axes[i].legend(loc='lower right', fontsize=8)
        axes[i].set_facecolor('#F8F9FA')

    # Macro in slot 9
    mean_fpr = np.linspace(0,1,200)
    mean_tpr = np.zeros(200)
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:,i], probs[:,i])
        mean_tpr += np.interp(mean_fpr, fpr, tpr)
    mean_tpr /= NUM_CLASSES
    macro_auc = auc(mean_fpr, mean_tpr)
    axes[9].plot(mean_fpr, mean_tpr, color='#E63946', lw=2.5,
                 label=f'Macro AUC={macro_auc:.3f}')
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:,i], probs[:,i])
        axes[9].plot(fpr, tpr, alpha=0.3, lw=1, color=CLASS_COLORS[i])
    axes[9].plot([0,1],[0,1],'k--', lw=1, alpha=0.4)
    axes[9].set_xlabel("FPR", fontsize=8); axes[9].set_ylabel("TPR", fontsize=8)
    axes[9].set_title("Macro Avg", fontweight='bold', fontsize=9)
    axes[9].legend(loc='lower right', fontsize=8)
    axes[9].set_facecolor('#F8F9FA')

    tag = "Phase 1" if "p1" in prefix else "Phase 2 (OOF)"
    fig.suptitle(f"Figure 5: ROC Curves — Per-Class & Macro ({tag})",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_roc_curves.png"), dpi=300)
    plt.close()
    print(f"  ✔ {prefix} ROC curves saved")
    return auc_vals, macro_auc


def save_fig_per_class_metrics(true, pred, out_dir, prefix="fig6"):
    rep = classification_report(true, pred, target_names=CLASS_NAMES,
                                 output_dict=True, zero_division=0)
    x     = np.arange(NUM_CLASSES)
    width = 0.26
    fig, ax = plt.subplots(figsize=(14, 6))
    for j, (m, c, h) in enumerate([
        ('precision','#457B9D',''),
        ('recall',   '#2A9D8F','///'),
        ('f1-score', '#E63946','...')
    ]):
        vals = [rep[cls][m] for cls in CLASS_NAMES]
        ax.bar(x + j*width, vals, width, label=m.capitalize(),
               color=c, alpha=0.85, edgecolor='white', hatch=h)

    ax.set_xticks(x + width)
    ax.set_xticklabels(CLASS_ABBR, rotation=30, ha='right')
    ax.set_xlabel("Lesion Class", fontweight='bold')
    ax.set_ylabel("Score",        fontweight='bold')
    ax.set_ylim(0, 1.18)
    macro_f1 = rep['macro avg']['f1-score']
    ax.axhline(macro_f1, color='black', ls='--', lw=1.5, alpha=0.6,
               label=f'Macro F1={macro_f1:.3f}')
    ax.legend(loc='upper right')
    ax.set_facecolor('#F8F9FA')

    tag = "Phase 1" if "p1" in prefix else "Phase 2 (OOF)"
    ax.set_title(f"Figure 6: Per-Class Precision / Recall / F1-Score — {tag}",
                 fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_per_class_metrics.png"), dpi=300)
    plt.close()
    print(f"  ✔ {prefix} per-class metrics saved")


def save_fig_sample_predictions(model, meta_df, out_dir, prefix="fig7",
                                 num_samples=16):
    model.eval()
    idx_list = np.random.choice(len(meta_df), num_samples, replace=False)
    ncols, nrows = 4, num_samples // 4
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows*3.2))
    axes = axes.flatten()

    for i, idx in enumerate(idx_list):
        row  = meta_df.iloc[idx]
        orig = np.array(Image.open(row['path']).convert("RGB"))
        aug  = val_transform(image=orig)['image']
        with torch.no_grad():
            out  = model(aug.unsqueeze(0).to(device))
            pred = out.argmax(1).item()
            conf = torch.softmax(out, 1).max().item()

        correct = (int(row['label']) == pred)
        color   = '#2A9D8F' if correct else '#E63946'
        sym     = '✓' if correct else '✗'
        axes[i].imshow(orig)
        axes[i].axis('off')
        axes[i].set_title(
            f"T: {CLASS_ABBR[int(row['label'])]}\n"
            f"P: {CLASS_ABBR[pred]} ({conf:.2f}) {sym}",
            fontsize=8, fontweight='bold', color=color, pad=3
        )
        rect = plt.Rectangle((0,0),1,1, transform=axes[i].transAxes,
                              lw=2.5, edgecolor=color, facecolor='none')
        axes[i].add_patch(rect)

    tag = "Phase 1" if "p1" in prefix else "Phase 2 (Best Fold)"
    fig.suptitle(f"Figure 7: Sample Predictions — {tag}  (✓ Correct / ✗ Incorrect)",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_sample_predictions.png"), dpi=300)
    plt.close()
    print(f"  ✔ {prefix} sample predictions saved")


def save_fig_results_table(true, pred, auc_vals, macro_auc, test_acc,
                            out_dir, prefix="fig8"):
    rep  = classification_report(true, pred, target_names=CLASS_NAMES,
                                  output_dict=True, zero_division=0)
    kappa = cohen_kappa_score(true, pred)
    rows = []
    for cls in CLASS_NAMES:
        rows.append([
            cls.replace('_',' ').title(),
            f"{rep[cls]['precision']:.4f}",
            f"{rep[cls]['recall']:.4f}",
            f"{rep[cls]['f1-score']:.4f}",
            f"{int(rep[cls]['support'])}",
            f"{auc_vals.get(cls,0):.4f}",
        ])

    col_labels = ['Class','Precision','Recall','F1-Score','Support','AUC-ROC']

    fig, ax = plt.subplots(figsize=(14, 5.5))
    ax.axis('off')
    tbl = ax.table(cellText=rows, colLabels=col_labels, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.85)

    for j in range(len(col_labels)):
        tbl[0,j].set_facecolor('#264653')
        tbl[0,j].set_text_props(color='white', fontweight='bold')
    for i in range(len(rows)):
        bg = '#F0F4F8' if i % 2 == 0 else 'white'
        for j in range(len(col_labels)):
            tbl[i+1,j].set_facecolor(bg)
    # Macro row
    tbl[len(rows)+1 if False else len(rows), 0]  # skip

    tag = "Phase 1" if "p1" in prefix else "Phase 2 (OOF)"
    ax.set_title(
        f"Figure 8: Classification Results — {tag}\n"
        f"Accuracy={test_acc:.4f}  |  Macro-AUC={macro_auc:.4f}"
        f"  |  Cohen's κ={kappa:.4f}  |  "
        f"Macro-F1={rep['macro avg']['f1-score']:.4f}",
        fontweight='bold', fontsize=11, pad=14
    )
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_results_table.png"), dpi=300)
    plt.close()
    print(f"  ✔ {prefix} results table saved")


# ╔══════════════════════════════════════════════════════════════╗
# ║                       PHASE 1                               ║
# ║              Single Train → All Figures                     ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "█"*65)
print("   PHASE 1 — SINGLE MODEL TRAINING")
print("█"*65)

# Fig 1 & 2 (dataset-level, done once)
save_fig1_distribution(train_meta_full, P1_DIR, "p1_fig1")
save_fig2_augmentation(P1_DIR, "p1_fig2")

# Train/Val split
train_df, val_df = train_test_split(
    train_meta_full, test_size=0.15,
    stratify=train_meta_full['label'], random_state=SEED
)
print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_meta)}")

BATCH = 24
train_ds = SkinDataset(train_df, train_transform)
val_ds   = SkinDataset(val_df,   val_transform)
test_ds  = SkinDataset(test_meta, val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH,
                          sampler=get_sampler(train_df),
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH,
                          shuffle=False, num_workers=2, pin_memory=True)

EPOCHS    = 30
p1_model  = SkinCancerModel(NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(weight=get_loss_weights(train_df))
optimizer = optim.AdamW(p1_model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

p1_history  = {'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[]}
p1_best_acc = 0.0
p1_best_wts = None

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_epoch(p1_model, train_loader, criterion, optimizer)
    val_true, val_pred, _ = evaluate_model(p1_model, val_loader)
    val_acc = accuracy_score(val_true, val_pred)

    p1_model.eval()
    v_loss = 0
    with torch.no_grad():
        for imgs, lbl in val_loader:
            imgs, lbl = imgs.to(device), lbl.to(device)
            v_loss += criterion(p1_model(imgs), lbl).item()
    v_loss /= len(val_loader)

    scheduler.step()
    p1_history['train_loss'].append(tr_loss)
    p1_history['train_acc'].append(tr_acc)
    p1_history['val_loss'].append(v_loss)
    p1_history['val_acc'].append(val_acc)

    flag = ''
    if val_acc > p1_best_acc:
        p1_best_acc = val_acc
        p1_best_wts = copy.deepcopy(p1_model.state_dict())
        torch.save(p1_best_wts, "phase1_best.pth")
        flag = ' ← ★ BEST'
    print(f"  Ep {epoch+1:02d}/{EPOCHS} | Loss:{tr_loss:.4f} "
          f"| TrainAcc:{tr_acc:.4f} | ValAcc:{val_acc:.4f}{flag}")

p1_model.load_state_dict(torch.load("phase1_best.pth"))
p1_true, p1_pred, p1_probs = evaluate_model(p1_model, test_loader)
p1_acc = accuracy_score(p1_true, p1_pred)

print(f"\nPhase 1 Test Accuracy : {p1_acc:.4f}")
print(classification_report(p1_true, p1_pred,
      target_names=CLASS_NAMES, digits=4, zero_division=0))

# Generate all Phase 1 figures
print("\n--- Phase 1 Figures ---")
save_fig_training_curves(p1_history, P1_DIR, "p1_fig3")
save_fig_confusion_matrix(p1_true, p1_pred, P1_DIR, "p1_fig4")
p1_auc_vals, p1_macro_auc = save_fig_roc_curves(p1_true, p1_probs, P1_DIR, "p1_fig5")
save_fig_per_class_metrics(p1_true, p1_pred, P1_DIR, "p1_fig6")
save_fig_sample_predictions(p1_model, test_meta, P1_DIR, "p1_fig7")
save_fig_results_table(p1_true, p1_pred, p1_auc_vals, p1_macro_auc, p1_acc,
                        P1_DIR, "p1_fig8")

# ╔══════════════════════════════════════════════════════════════╗
# ║                       PHASE 2                               ║
# ║              5-Fold CV → All Figures + OOF                  ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "█"*65)
print("   PHASE 2 — 5-FOLD STRATIFIED CROSS-VALIDATION")
print("█"*65)

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_histories = []
fold_results   = []
oof_true, oof_pred, oof_probs = [], [], []

X = train_meta_full.index.values
y = train_meta_full['label'].values

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n{'─'*55}")
    print(f"  FOLD {fold+1}/{N_FOLDS}")
    print(f"{'─'*55}")

    fold_train = train_meta_full.iloc[tr_idx].reset_index(drop=True)
    fold_val   = train_meta_full.iloc[val_idx].reset_index(drop=True)

    tr_ds  = SkinDataset(fold_train, train_transform)
    vl_ds  = SkinDataset(fold_val,   val_transform)

    tr_ld  = DataLoader(tr_ds, batch_size=BATCH,
                        sampler=get_sampler(fold_train),
                        num_workers=2, pin_memory=True)
    vl_ld  = DataLoader(vl_ds, batch_size=BATCH,
                        shuffle=False, num_workers=2, pin_memory=True)

    fold_model  = SkinCancerModel(NUM_CLASSES).to(device)
    fold_crit   = nn.CrossEntropyLoss(weight=get_loss_weights(fold_train))
    fold_optim  = optim.AdamW(fold_model.parameters(), lr=2e-4, weight_decay=1e-4)
    fold_sched  = optim.lr_scheduler.CosineAnnealingLR(fold_optim, T_max=EPOCHS, eta_min=1e-6)

    fhist     = {'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[]}
    best_acc  = 0.0
    best_wts  = None

    for epoch in range(EPOCHS):
        tr_loss, tr_acc = train_epoch(fold_model, tr_ld, fold_crit, fold_optim)
        vt, vp, _       = evaluate_model(fold_model, vl_ld)
        val_acc         = accuracy_score(vt, vp)

        fold_model.eval()
        v_loss = 0
        with torch.no_grad():
            for imgs, lbl in vl_ld:
                imgs, lbl = imgs.to(device), lbl.to(device)
                v_loss += fold_crit(fold_model(imgs), lbl).item()
        v_loss /= len(vl_ld)

        fold_sched.step()
        fhist['train_loss'].append(tr_loss)
        fhist['train_acc'].append(tr_acc)
        fhist['val_loss'].append(v_loss)
        fhist['val_acc'].append(val_acc)

        flag = ''
        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = copy.deepcopy(fold_model.state_dict())
            flag     = ' ★'
        print(f"    Ep {epoch+1:02d} | Loss:{tr_loss:.4f} "
              f"| Val:{val_acc:.4f}{flag}")

    fold_model.load_state_dict(best_wts)
    torch.save(best_wts, f"fold{fold+1}_best.pth")

    vt, vp, vprob = evaluate_model(fold_model, vl_ld)
    rep = classification_report(vt, vp, output_dict=True, zero_division=0)
    fold_results.append({
        'fold':        fold+1,
        'accuracy':    accuracy_score(vt, vp),
        'macro_f1':    rep['macro avg']['f1-score'],
        'weighted_f1': rep['weighted avg']['f1-score'],
        'kappa':       cohen_kappa_score(vt, vp),
    })
    fold_histories.append(fhist)
    oof_true.extend(vt.tolist())
    oof_pred.extend(vp.tolist())
    oof_probs.extend(vprob.tolist())
    print(f"\n  Fold {fold+1} → Acc:{fold_results[-1]['accuracy']:.4f}"
          f"  MacroF1:{fold_results[-1]['macro_f1']:.4f}"
          f"  κ:{fold_results[-1]['kappa']:.4f}")

oof_true  = np.array(oof_true)
oof_pred  = np.array(oof_pred)
oof_probs = np.array(oof_probs)

print("\n" + "═"*65)
print("  5-FOLD CV SUMMARY")
print("═"*65)
for r in fold_results:
    print(f"  Fold {r['fold']}: Acc={r['accuracy']:.4f}  "
          f"MacroF1={r['macro_f1']:.4f}  κ={r['kappa']:.4f}")
mean_acc = np.mean([r['accuracy']  for r in fold_results])
std_acc  = np.std( [r['accuracy']  for r in fold_results])
mean_f1  = np.mean([r['macro_f1']  for r in fold_results])
std_f1   = np.std( [r['macro_f1']  for r in fold_results])
print(f"\n  Mean Accuracy : {mean_acc:.4f} ± {std_acc:.4f}")
print(f"  Mean Macro-F1 : {mean_f1:.4f} ± {std_f1:.4f}")

# Phase 2 figures
print("\n--- Phase 2 Figures ---")

# Fig 3: Mean training curves across folds
max_ep = max(len(h['train_loss']) for h in fold_histories)
def pad(lst): return lst + [lst[-1]]*(max_ep-len(lst))
mean_hist = {
    'train_loss': np.mean([pad(h['train_loss']) for h in fold_histories], 0),
    'train_acc':  np.mean([pad(h['train_acc'])  for h in fold_histories], 0),
    'val_loss':   np.mean([pad(h['val_loss'])   for h in fold_histories], 0),
    'val_acc':    np.mean([pad(h['val_acc'])    for h in fold_histories], 0),
}
save_fig_training_curves(mean_hist, P2_DIR, "p2_fig3")

# Fig 3b: All fold curves overlaid
ep_x = np.arange(1, max_ep+1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for i, h in enumerate(fold_histories):
    axes[0].plot(ep_x, pad(h['train_loss']), alpha=0.3, lw=1.2,
                 color=CLASS_COLORS[i], label=f'Fold {i+1}')
    axes[1].plot(ep_x, pad(h['val_acc']),   alpha=0.3, lw=1.2,
                 color=CLASS_COLORS[i], label=f'Fold {i+1}')
tr_loss_arr = np.array([pad(h['train_loss']) for h in fold_histories])
va_arr      = np.array([pad(h['val_acc'])    for h in fold_histories])
axes[0].plot(ep_x, tr_loss_arr.mean(0), color='black', lw=2.5, label='Mean')
axes[0].fill_between(ep_x,
    tr_loss_arr.mean(0)-tr_loss_arr.std(0),
    tr_loss_arr.mean(0)+tr_loss_arr.std(0),
    alpha=0.15, color='black')
axes[1].plot(ep_x, va_arr.mean(0), color='black', lw=2.5, label='Mean')
axes[1].fill_between(ep_x,
    va_arr.mean(0)-va_arr.std(0),
    va_arr.mean(0)+va_arr.std(0),
    alpha=0.15, color='black')
for ax, label in zip(axes, ['Train Loss','Val Accuracy']):
    ax.set_xlabel("Epoch", fontweight='bold')
    ax.set_ylabel(label,   fontweight='bold')
    ax.set_title(label,    fontweight='bold')
    ax.legend(ncol=3, fontsize=8)
    ax.set_facecolor('#F8F9FA')
fig.suptitle("Figure 3b: All-Fold Training Curves with Mean ± SD",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(P2_DIR, "p2_fig3b_all_fold_curves.png"), dpi=300)
plt.close()
print("  ✔ p2_fig3b all fold curves saved")

save_fig_confusion_matrix(oof_true, oof_pred, P2_DIR, "p2_fig4")
p2_auc_vals, p2_macro_auc = save_fig_roc_curves(oof_true, oof_probs, P2_DIR, "p2_fig5")
save_fig_per_class_metrics(oof_true, oof_pred, P2_DIR, "p2_fig6")

# Fig 7: CV summary boxplot
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
metrics_keys   = ['accuracy','macro_f1','weighted_f1','kappa']
metrics_labels = ['Accuracy','Macro F1','Weighted F1','Cohen κ']
for ax, key, label in zip(axes, metrics_keys, metrics_labels):
    vals = [r[key] for r in fold_results]
    bp   = ax.boxplot(vals, patch_artist=True, widths=0.5,
                      medianprops={'color':'#E63946','linewidth':2.5})
    bp['boxes'][0].set_facecolor('#457B9D'); bp['boxes'][0].set_alpha(0.6)
    for vi, (v, c) in enumerate(zip(vals, CLASS_COLORS)):
        ax.scatter(1, v, color=c, zorder=5, s=70, edgecolors='black', lw=0.8)
        ax.annotate(f'F{vi+1}:{v:.3f}', (1.13, v), fontsize=8, va='center',
                    color=c, fontweight='bold')
    m, s = np.mean(vals), np.std(vals)
    ax.set_title(f"{label}\n{m:.4f} ± {s:.4f}", fontweight='bold', fontsize=10)
    ax.set_xticks([])
    ax.set_xlim(0.7, 1.55)
    ax.set_facecolor('#F8F9FA')
fig.suptitle("Figure 7: 5-Fold Cross-Validation Performance Summary",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(P2_DIR, "p2_fig7_cv_summary.png"), dpi=300)
plt.close()
print("  ✔ p2_fig7 CV summary saved")

# Best fold model for sample predictions & results table
best_fold = int(np.argmax([r['macro_f1'] for r in fold_results]))
best_fold_model = SkinCancerModel(NUM_CLASSES).to(device)
best_fold_model.load_state_dict(torch.load(f"fold{best_fold+1}_best.pth"))
test_true2, test_pred2, test_probs2 = evaluate_model(best_fold_model, test_loader)
test_acc2 = accuracy_score(test_true2, test_pred2)
print(f"\nBest Fold {best_fold+1} Test Accuracy: {test_acc2:.4f}")

save_fig_sample_predictions(best_fold_model, test_meta, P2_DIR, "p2_fig8")
save_fig_results_table(test_true2, test_pred2, p2_auc_vals, p2_macro_auc, test_acc2,
                        P2_DIR, "p2_fig9")

# Save CV JSON
with open(os.path.join(P2_DIR, "cv_results.json"), "w") as f:
    json.dump({
        'fold_results': fold_results,
        'mean_accuracy': mean_acc, 'std_accuracy': std_acc,
        'mean_macro_f1': mean_f1,  'std_macro_f1': std_f1,
        'oof_accuracy': float(accuracy_score(oof_true, oof_pred)),
    }, f, indent=2)

# ╔══════════════════════════════════════════════════════════════╗
# ║                  PHASE 3 — EXPLAINABLE AI                   ║
# ║    Grad-CAM  ·  Score-CAM  ·  Feature Importance (SHAP)     ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "█"*65)
print("   PHASE 3 — EXPLAINABLE AI (XAI)")
print("█"*65)

import cv2

# ─────────────────────────────────────────────────────────────
# Grad-CAM (pure PyTorch, no extra library needed)
# ─────────────────────────────────────────────────────────────
class GradCAM:
    def __init__(self, model, target_layer):
        self.model       = model
        self.target_layer= target_layer
        self.gradients   = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(1).item()
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, class_idx] = 1
        output.backward(gradient=one_hot)
        weights = self.gradients.mean(dim=[2,3], keepdim=True)
        cam     = (weights * self.activations).sum(dim=1, keepdim=True)
        cam     = F.relu(cam)
        cam     = F.interpolate(cam, size=(224,224),
                                mode='bilinear', align_corners=False)
        cam     = cam.squeeze().cpu().numpy()
        cam    -= cam.min()
        if cam.max() > 0:
            cam /= cam.max()
        return cam, class_idx


def overlay_gradcam(orig_img_np, cam, alpha=0.45):
    heatmap = cv2.applyColorMap(
        np.uint8(255 * cam), cv2.COLORMAP_JET
    )
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    heatmap = cv2.resize(heatmap, (orig_img_np.shape[1], orig_img_np.shape[0]))
    overlay = (alpha * heatmap + (1-alpha) * orig_img_np).astype(np.uint8)
    return overlay


# ─────────────────────────────────────────────────────────────
# XAI Figure 1: Grad-CAM Grid (2 samples per class)
# ─────────────────────────────────────────────────────────────
# Register Grad-CAM on last conv block of EfficientNet-B2
grad_cam = GradCAM(
    best_fold_model,
    best_fold_model.backbone.features[-1]
)

print("\nGenerating Grad-CAM visualizations...")
samples_per_class = 2
nrows = NUM_CLASSES
ncols = 1 + samples_per_class * 2    # label + orig + cam per sample

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*2.4, nrows*2.6))

for r, cls_name in enumerate(CLASS_NAMES):
    cls_samples = test_meta[test_meta['class'] == cls_name].head(samples_per_class)
    axes[r][0].axis('off')
    axes[r][0].text(0.5, 0.5, cls_name.replace('_','\n'),
                    ha='center', va='center', fontsize=8,
                    fontweight='bold', transform=axes[r][0].transAxes,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor=CLASS_COLORS[r],
                              alpha=0.3, edgecolor='none'))

    for s_idx, (_, row) in enumerate(cls_samples.iterrows()):
        orig  = np.array(Image.open(row['path']).convert("RGB"))
        aug   = val_transform(image=orig)['image'].unsqueeze(0).to(device)
        cam, pred_idx = grad_cam.generate(aug)
        overlay = overlay_gradcam(
            cv2.resize(orig, (224,224)), cam
        )
        col_orig = 1 + s_idx*2
        col_cam  = 2 + s_idx*2

        axes[r][col_orig].imshow(cv2.resize(orig, (224,224)))
        axes[r][col_orig].axis('off')
        if r == 0:
            axes[r][col_orig].set_title(f'Original {s_idx+1}',
                                         fontsize=8, fontweight='bold')

        axes[r][col_cam].imshow(overlay)
        axes[r][col_cam].axis('off')
        correct = (int(row['label']) == pred_idx)
        color   = '#2A9D8F' if correct else '#E63946'
        if r == 0:
            axes[r][col_cam].set_title(f'Grad-CAM {s_idx+1}',
                                        fontsize=8, fontweight='bold')
        axes[r][col_cam].set_title(
            axes[r][col_cam].get_title() + f'\nP:{CLASS_ABBR[pred_idx]}',
            fontsize=7, color=color, fontweight='bold'
        )

fig.suptitle("XAI Figure 1: Grad-CAM Visualizations — All 9 Classes (Best Fold Model)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(XAI_DIR, "xai_fig1_gradcam_all_classes.png"), dpi=300)
plt.close()
print("  ✔ XAI Fig 1: Grad-CAM all classes saved")

# ─────────────────────────────────────────────────────────────
# XAI Figure 2: Grad-CAM Correct vs Wrong comparison
# ─────────────────────────────────────────────────────────────
print("Generating Grad-CAM correct vs wrong comparison...")

best_fold_model.eval()
correct_samples = []
wrong_samples   = []

for _, row in test_meta.iterrows():
    if len(correct_samples) >= 4 and len(wrong_samples) >= 4:
        break
    orig = np.array(Image.open(row['path']).convert("RGB"))
    aug  = val_transform(image=orig)['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        out  = best_fold_model(aug)
        pred = out.argmax(1).item()
        conf = torch.softmax(out,1).max().item()
    correct = (int(row['label']) == pred)
    entry = {'row': row, 'pred': pred, 'conf': conf}
    if correct and len(correct_samples) < 4:
        correct_samples.append(entry)
    elif not correct and len(wrong_samples) < 4:
        wrong_samples.append(entry)

fig, axes = plt.subplots(4, 6, figsize=(16, 11))
col_labels = ['Original','Grad-CAM','','Original','Grad-CAM','']

for r in range(4):
    for grp, grp_samples, col_offset, grp_label in [
        ('Correct', correct_samples, 0, '#2A9D8F'),
        ('Wrong',   wrong_samples,   3, '#E63946')
    ]:
        if r >= len(grp_samples): break
        entry = grp_samples[r]
        row   = entry['row']
        orig  = np.array(Image.open(row['path']).convert("RGB"))
        aug   = val_transform(image=orig)['image'].unsqueeze(0).to(device)
        cam, _ = grad_cam.generate(aug, class_idx=int(row['label']))
        overlay = overlay_gradcam(cv2.resize(orig,(224,224)), cam)

        axes[r][col_offset].imshow(cv2.resize(orig,(224,224)))
        axes[r][col_offset].axis('off')
        axes[r][col_offset+1].imshow(overlay)
        axes[r][col_offset+1].axis('off')
        axes[r][col_offset+2].axis('off')

        true_name = CLASS_ABBR[int(row['label'])]
        pred_name = CLASS_ABBR[entry['pred']]
        axes[r][col_offset].set_ylabel(
            f"T:{true_name}\nP:{pred_name}\n({entry['conf']:.2f})",
            fontsize=8, color=grp_label, fontweight='bold',
            rotation=0, labelpad=55, va='center'
        )

for c, lbl in enumerate(['Orig (✓)','CAM (✓)','','Orig (✗)','CAM (✗)','']):
    if lbl:
        axes[0][c].set_title(lbl, fontsize=10, fontweight='bold',
                              color='#2A9D8F' if '✓' in lbl else '#E63946')

fig.suptitle("XAI Figure 2: Grad-CAM — Correct Predictions vs Misclassifications",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(XAI_DIR, "xai_fig2_gradcam_correct_vs_wrong.png"), dpi=300)
plt.close()
print("  ✔ XAI Fig 2: Correct vs wrong Grad-CAM saved")

# ─────────────────────────────────────────────────────────────
# XAI Figure 3: Class Activation Statistics
# (Average Grad-CAM intensity per predicted class)
# ─────────────────────────────────────────────────────────────
print("Computing class-wise Grad-CAM intensity statistics...")
class_cam_means = {cls: [] for cls in CLASS_NAMES}
class_cam_stds  = {cls: [] for cls in CLASS_NAMES}

sample_size = min(30, len(test_meta) // NUM_CLASSES)
for cls_name in CLASS_NAMES:
    cls_samples = test_meta[test_meta['class'] == cls_name].head(sample_size)
    for _, row in cls_samples.iterrows():
        try:
            orig = np.array(Image.open(row['path']).convert("RGB"))
            aug  = val_transform(image=orig)['image'].unsqueeze(0).to(device)
            cam, _ = grad_cam.generate(aug)
            class_cam_means[cls_name].append(cam.mean())
            class_cam_stds[cls_name].append(cam.std())
        except:
            pass

means = [np.mean(class_cam_means[c]) if class_cam_means[c] else 0
         for c in CLASS_NAMES]
stds  = [np.std(class_cam_means[c]) if class_cam_means[c] else 0
         for c in CLASS_NAMES]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars = axes[0].bar(CLASS_ABBR, means, yerr=stds,
                   color=CLASS_COLORS, edgecolor='white', lw=0.8,
                   capsize=5, alpha=0.9)
axes[0].set_xlabel("Lesion Class", fontweight='bold')
axes[0].set_ylabel("Mean Grad-CAM Activation", fontweight='bold')
axes[0].set_title("(a) Mean Grad-CAM Intensity per Class", fontweight='bold')
axes[0].set_facecolor('#F8F9FA')
axes[0].tick_params(axis='x', rotation=30)

# Radar chart of CAM intensity
angles = np.linspace(0, 2*np.pi, NUM_CLASSES, endpoint=False).tolist()
angles += angles[:1]
vals   = means + [means[0]]

ax_radar = plt.subplot(1,2,2, polar=True)
ax_radar.plot(angles, vals, 'o-', linewidth=2, color='#E63946')
ax_radar.fill(angles, vals, alpha=0.15, color='#E63946')
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(CLASS_ABBR, fontsize=9, fontweight='bold')
ax_radar.set_title("(b) Grad-CAM Activation Radar", fontweight='bold', pad=15)
ax_radar.set_facecolor('#F8F9FA')

fig.suptitle("XAI Figure 3: Class-Wise Grad-CAM Activation Statistics",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(XAI_DIR, "xai_fig3_gradcam_class_statistics.png"), dpi=300)
plt.close()
print("  ✔ XAI Fig 3: CAM statistics saved")

# ─────────────────────────────────────────────────────────────
# XAI Figure 4: Prediction Confidence Distribution
# ─────────────────────────────────────────────────────────────
print("Generating confidence distribution plot...")
all_confs = test_probs2.max(axis=1)  # from phase 2 best fold

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Overall histogram
axes[0].hist(all_confs, bins=30, color='#457B9D', edgecolor='white', alpha=0.85)
axes[0].axvline(all_confs.mean(), color='#E63946', lw=2, ls='--',
                label=f'Mean={all_confs.mean():.3f}')
axes[0].set_xlabel("Prediction Confidence", fontweight='bold')
axes[0].set_ylabel("Frequency",             fontweight='bold')
axes[0].set_title("(a) Overall Confidence Distribution", fontweight='bold')
axes[0].legend()
axes[0].set_facecolor('#F8F9FA')

# Per-class violin
conf_by_class = [
    test_probs2[test_true2 == i].max(axis=1)
    for i in range(NUM_CLASSES)
]
parts = axes[1].violinplot(
    [c for c in conf_by_class if len(c)>0],
    positions=range(NUM_CLASSES),
    showmeans=True, showmedians=True
)
for pc, color in zip(parts['bodies'], CLASS_COLORS):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
axes[1].set_xticks(range(NUM_CLASSES))
axes[1].set_xticklabels(CLASS_ABBR, rotation=30, ha='right')
axes[1].set_xlabel("Lesion Class", fontweight='bold')
axes[1].set_ylabel("Confidence",   fontweight='bold')
axes[1].set_title("(b) Per-Class Confidence Distribution", fontweight='bold')
axes[1].set_facecolor('#F8F9FA')

fig.suptitle("XAI Figure 4: Model Prediction Confidence Analysis",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(XAI_DIR, "xai_fig4_confidence_distribution.png"), dpi=300)
plt.close()
print("  ✔ XAI Fig 4: Confidence distribution saved")

# ─────────────────────────────────────────────────────────────
# XAI Figure 5: Top-K Softmax Probability Bars (per sample)
# ─────────────────────────────────────────────────────────────
print("Generating top-K probability plots...")
n_samples = 8
idx_list  = np.random.choice(len(test_meta), n_samples, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, idx in enumerate(idx_list):
    row   = test_meta.iloc[idx]
    probs = test_probs2[idx]
    top_k = np.argsort(probs)[::-1][:5]

    colors_bar = ['#E63946' if j == int(row['label']) else '#B0BEC5'
                  for j in top_k]
    axes[i].barh(
        [CLASS_ABBR[j] for j in top_k[::-1]],
        [probs[j]      for j in top_k[::-1]],
        color=colors_bar[::-1], edgecolor='white', alpha=0.9
    )
    axes[i].set_xlim(0, 1)
    axes[i].set_xlabel("Probability", fontsize=8)
    correct = (int(row['label']) == np.argmax(probs))
    color   = '#2A9D8F' if correct else '#E63946'
    sym     = '✓' if correct else '✗'
    axes[i].set_title(
        f"True: {CLASS_ABBR[int(row['label'])]} {sym}",
        fontsize=9, fontweight='bold', color=color
    )
    axes[i].set_facecolor('#F8F9FA')
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)

fig.suptitle("XAI Figure 5: Top-5 Class Probabilities per Test Sample (red=true class)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(XAI_DIR, "xai_fig5_topk_probabilities.png"), dpi=300)
plt.close()
print("  ✔ XAI Fig 5: Top-K probabilities saved")

# ─────────────────────────────────────────────────────────────
# XAI Figure 6: Feature Embedding Visualization (t-SNE / UMAP)
# ─────────────────────────────────────────────────────────────
try:
    from sklearn.manifold import TSNE

    print("Extracting features for t-SNE embedding...")
    best_fold_model.eval()
    all_feats, all_lbls = [], []

    feat_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                             num_workers=2, pin_memory=True)
    with torch.no_grad():
        for imgs, lbls in feat_loader:
            imgs = imgs.to(device)
            feats = best_fold_model.get_features(imgs)
            all_feats.extend(feats.cpu().numpy())
            all_lbls.extend(lbls.numpy())

    all_feats = np.array(all_feats)
    all_lbls  = np.array(all_lbls)

    # t-SNE
    print("Running t-SNE (may take 1-2 min)...")
    tsne      = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
    embedded  = tsne.fit_transform(all_feats)

    fig, ax = plt.subplots(figsize=(10, 8))
    for i, cls in enumerate(CLASS_NAMES):
        mask = all_lbls == i
        ax.scatter(embedded[mask,0], embedded[mask,1],
                   c=CLASS_COLORS[i], label=CLASS_ABBR[i],
                   s=18, alpha=0.75, edgecolors='white', linewidths=0.3)
    ax.set_xlabel("t-SNE Dim 1", fontweight='bold')
    ax.set_ylabel("t-SNE Dim 2", fontweight='bold')
    ax.set_title("XAI Figure 6: t-SNE Feature Space — 9-Class Skin Lesions",
                 fontweight='bold', fontsize=13)
    ax.legend(title="Class", bbox_to_anchor=(1.02,1), loc='upper left',
              fontsize=9, title_fontsize=10)
    ax.set_facecolor('#F8F9FA')
    plt.tight_layout()
    plt.savefig(os.path.join(XAI_DIR, "xai_fig6_tsne_embedding.png"), dpi=300)
    plt.close()
    print("  ✔ XAI Fig 6: t-SNE embedding saved")

except Exception as e:
    print(f"  ⚠ t-SNE skipped: {e}")

# ─────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────────────────────
print("\n" + "═"*65)
print("  COMPLETE PIPELINE FINISHED")
print("═"*65)
print(f"\n  Phase 1 Test Accuracy : {p1_acc:.4f}")
print(f"  Phase 1 Macro-AUC    : {p1_macro_auc:.4f}")
print(f"\n  Phase 2 CV Mean Acc  : {mean_acc:.4f} ± {std_acc:.4f}")
print(f"  Phase 2 CV Macro-F1  : {mean_f1:.4f} ± {std_f1:.4f}")
print(f"\n  Output dirs:")
print(f"    Phase 1 figures → {P1_DIR}/")
print(f"    Phase 2 figures → {P2_DIR}/")
print(f"    XAI figures     → {XAI_DIR}/")

for folder, label in [(P1_DIR,'Phase 1'), (P2_DIR,'Phase 2'), (XAI_DIR,'XAI')]:
    figs = sorted(os.listdir(folder))
    print(f"\n  {label} ({len(figs)} files):")
    for f in figs:
        print(f"    {f}")

Using Device: cuda

9 Classes detected: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']

Total Train: 2239 | Test: 118

Train class distribution:
class
pigmented benign keratosis    462
melanoma                      438
basal cell carcinoma          376
nevus                         357
squamous cell carcinoma       181
vascular lesion               139
actinic keratosis             114
dermatofibroma                 95
seborrheic keratosis           77
Name: count, dtype: int64

█████████████████████████████████████████████████████████████████
   PHASE 1 — SINGLE MODEL TRAINING
█████████████████████████████████████████████████████████████████
  ✔ p1_fig1 class distribution saved
  ✔ p1_fig2 augmentation grid saved

Train: 1903 | Val: 336 | Test: 118
Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.ca

100%|██████████| 35.2M/35.2M [00:00<00:00, 175MB/s] 


  Ep 01/30 | Loss:1.5080 | TrainAcc:0.3573 | ValAcc:0.3958 ← ★ BEST
  Ep 02/30 | Loss:0.9837 | TrainAcc:0.5218 | ValAcc:0.5000 ← ★ BEST
  Ep 03/30 | Loss:0.8062 | TrainAcc:0.5765 | ValAcc:0.5595 ← ★ BEST
  Ep 04/30 | Loss:0.6921 | TrainAcc:0.6395 | ValAcc:0.5595
  Ep 05/30 | Loss:0.6350 | TrainAcc:0.6521 | ValAcc:0.5565
  Ep 06/30 | Loss:0.5571 | TrainAcc:0.6900 | ValAcc:0.6012 ← ★ BEST
  Ep 07/30 | Loss:0.5090 | TrainAcc:0.7168 | ValAcc:0.6161 ← ★ BEST
  Ep 08/30 | Loss:0.4878 | TrainAcc:0.7204 | ValAcc:0.6220 ← ★ BEST
  Ep 09/30 | Loss:0.4394 | TrainAcc:0.7535 | ValAcc:0.5982
  Ep 10/30 | Loss:0.4304 | TrainAcc:0.7530 | ValAcc:0.6577 ← ★ BEST
  Ep 11/30 | Loss:0.3797 | TrainAcc:0.7814 | ValAcc:0.6577
  Ep 12/30 | Loss:0.4107 | TrainAcc:0.7662 | ValAcc:0.6696 ← ★ BEST
  Ep 13/30 | Loss:0.3890 | TrainAcc:0.7709 | ValAcc:0.6786 ← ★ BEST
  Ep 14/30 | Loss:0.3488 | TrainAcc:0.8066 | ValAcc:0.6696
  Ep 15/30 | Loss:0.3480 | TrainAcc:0.7924 | ValAcc:0.6875 ← ★ BEST
  Ep 16/30 | Loss:0.2839 